# Hydrogen properties comparison

Comparación de la conductividad térmica de H₂ mediante Assael et al. (2011)
y la aproximación de Sutherland documentada en COMSOL Multiphysics (FEM).
Este notebook es independiente del simulador: no modifica sus propiedades.

Incluye densidad (Leachman) y viscosidad (Muzny), con comparaciones de ingeniería.


## Modelos y fuentes

**Assael et al. (2011):** se evalúa únicamente el término de gas diluido para hidrógeno normal,
ecuación 2 y tabla 2 (p. 5), no la correlación completa a alta presión:

\[
k_A(T)=\frac{\sum_{i=0}^{6} A_{1,i}(T/33.145)^i}{\sum_{i=0}^{3} A_{2,i}(T/33.145)^i}.
\]

Se omiten los términos de densidad y de incremento crítico. La comparación corresponde
al límite de baja densidad, no a una presión exacta especificada. El rango global publicado
termina en 1000 K; se sombrea la extensión por encima de ese valor. El término diluido se
ajustó con cálculos teóricos hasta 2000 K (sección 3.1).
[Artículo de referencia alojado por NIST](https://www.nist.gov/system/files/documents/srd/jpcrd402011Corr.pdf).

**Sutherland (aproximación de ingeniería en COMSOL):**

\[
k_S(T)=k_0\left(\frac{T}{T_0}\right)^{3/2}\frac{T_0+S_k}{T+S_k},
\qquad k_0=0.168\;\mathrm{W/(m\,K)},\quad T_0=273\;\mathrm K,\quad S_k=120\;\mathrm K.
\]

Los parámetros son los de H₂ en la tabla 5-3 y la fórmula es la ecuación 5-10 del
[manual de COMSOL](https://doc.comsol.com/6.3/doc/com.comsol.help.cfd/cfd_ug_fluidflow_high_mach.08.43.html).
La documentación no especifica un intervalo de precisión para H₂: graficar la fórmula
no valida su exactitud en todo el intervalo. Es una correlación de propiedades que se
puede usar en FEM, no una solución por elementos finitos.

En ambas expresiones, T está en kelvin y k en W/(m·K).


## Settings
Ajusta el intervalo, los puntos de la tabla y el tamaño de la gráfica aquí.

In [1]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

settings = {
    "temperature_min_K": 273.0,
    "temperature_max_K": 1100.0,
    "samples": 500,
    "table_temperatures_K": [273, 300, 400, 600, 800, 1000, 1100],
    "height": 850,
}
sutherland_parameters = {"k0": 0.168, "T0": 273.0, "Sk": 120.0}

# Assael et al. (2011), Table 2: ascending powers of T/Tc.
ASSAEL_NUMERATOR = [-0.340976, 4.58820, -1.45080, 0.326394,
                    0.00316939, 0.000190592, -0.000001139]
ASSAEL_DENOMINATOR = [138.497, -22.1878, 4.57151, 1.0]


def positive_temperature(temperature):
    temperature = np.asarray(temperature, dtype=float)
    if np.any(~np.isfinite(temperature)) or np.any(temperature <= 0):
        raise ValueError("Temperatures must be finite and positive, in kelvin.")
    return temperature


def conductivity_assael(temperature):
    reduced = positive_temperature(temperature) / 33.145
    return (np.polynomial.polynomial.polyval(reduced, ASSAEL_NUMERATOR)
            / np.polynomial.polynomial.polyval(reduced, ASSAEL_DENOMINATOR))


def conductivity_sutherland(temperature):
    temperature = positive_temperature(temperature)
    k0, T0, Sk = (sutherland_parameters[key] for key in ["k0", "T0", "Sk"])
    return k0 * (temperature / T0)**1.5 * (T0 + Sk) / (temperature + Sk)


if not 100 <= settings["temperature_min_K"] < settings["temperature_max_K"] <= 2000:
    raise ValueError("Choose 100 <= Tmin < Tmax <= 2000 K for this comparison.")
if settings["samples"] < 2:
    raise ValueError("At least two samples are required.")
temperature_K = np.linspace(settings["temperature_min_K"],
                            settings["temperature_max_K"], settings["samples"])
k_assael = conductivity_assael(temperature_K)
k_sutherland = conductivity_sutherland(temperature_K)
relative_difference = 100 * (k_sutherland / k_assael - 1)

layout = {
    "title": {"text": "Conductividad del hidrógeno: Assael y Sutherland",
              "x": 0.5, "xanchor": "center"},
    "height": settings["height"],
    "template": "plotly_white",
    "font": {"family": "Arial", "size": 15},
    "margin": {"t": 140, "b": 70, "l": 95, "r": 50},
    "legend": {"orientation": "h", "y": 1.10, "x": 0},
    "hovermode": "x unified",
}


## Plot
El panel inferior muestra `100 × (Sutherland − Assael) / Assael`; es diferencia entre modelos, no error experimental.

In [2]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True,
                    row_heights=[0.68, 0.32], vertical_spacing=0.12)
fig.add_trace(go.Scatter(x=temperature_K, y=k_assael, mode="lines",
                        name="Assael et al. (2011), gas diluido",
                        line={"color": "#1f77b4", "width": 3}), row=1, col=1)
fig.add_trace(go.Scatter(x=temperature_K, y=k_sutherland, mode="lines",
                        name="Sutherland — COMSOL",
                        line={"color": "#d95f02", "width": 3, "dash": "dash"}), row=1, col=1)
fig.add_trace(go.Scatter(x=temperature_K, y=relative_difference, mode="lines",
                        name="Diferencia relativa", showlegend=False,
                        line={"color": "#6a3d9a", "width": 2}), row=2, col=1)
fig.update_layout(**layout)
fig.update_yaxes(title_text="k [W/(m·K)]", row=1, col=1)
fig.update_yaxes(title_text="Diferencia [%]", row=2, col=1)
fig.update_xaxes(title_text="Temperatura [K]", row=2, col=1)
fig.add_hline(y=0, line_color="#888888", line_dash="dot", row=2, col=1)
if settings["temperature_max_K"] > 1000:
    fig.add_vrect(x0=max(1000, settings["temperature_min_K"]),
                  x1=settings["temperature_max_K"], fillcolor="#ffc857",
                  opacity=0.18, line_width=0, row="all", col=1)
    fig.add_annotation(x=1, y=1.02, xref="paper", yref="paper",
                       text="Sombreado: T > 1000 K", showarrow=False, xanchor="right")
fig.show()


Plotly figure

## Valores de comparación

In [3]:
table_T = positive_temperature(settings["table_temperatures_K"])
table_A = conductivity_assael(table_T)
table_S = conductivity_sutherland(table_T)
table_percent = 100 * (table_S / table_A - 1)
print(f"{'T [K]':>8} {'Assael [W/mK]':>16} {'Sutherland [W/mK]':>20} {'Diferencia [%]':>17}")
for T, a, s, difference in zip(table_T, table_A, table_S, table_percent):
    print(f"{T:8.1f} {a:16.5f} {s:20.5f} {difference:17.2f}")
print("\nDiferencias respecto a Assael; no son incertidumbres ni errores frente a mediciones.")


   T [K]    Assael [W/mK]    Sutherland [W/mK]    Diferencia [%]
   273.0          0.17324              0.16800             -3.02
   300.0          0.18656              0.18109             -2.93
   400.0          0.23098              0.22519             -2.51
   600.0          0.30895              0.29878             -3.29
   800.0          0.38380              0.36000             -6.20
  1000.0          0.46032              0.41328            -10.22
  1100.0          0.49965              0.43771            -12.40

Diferencias respecto a Assael; no son incertidumbres ni errores frente a mediciones.


## Densidad y viscosidad: modelos de referencia

Para hidrógeno normal se seleccionan las formulaciones de referencia de **Leachman et al.
(2009)** para densidad y **Muzny et al. (2013, corrección de 2022)** para viscosidad.
Son las fuentes de propiedades identificadas por [CoolProp](https://coolprop.org/fluid_properties/fluids/Hydrogen.html),
cuya implementación HEOS se utiliza aquí; no son ajustes hechos a nuestra simulación.

La ecuación de estado de Leachman utiliza energía de Helmholtz reducida:
\[
p=\rho R_sT\left[1+\delta\left(\frac{\partial\alpha^r}{\partial\delta}\right)_\tau\right],
\qquad \delta=\rho/\rho_c,\quad \tau=T_c/T.
\]
CoolProp resuelve la densidad para cada par (T, p). Se compara con
\(\rho_{ideal}=pM/(RT)\). Su concordancia a baja presión no implica que el gas
ideal sea equivalente al modelo de referencia a presiones elevadas.

Muzny expresa la viscosidad como suma de contribuciones diluida, inicial en densidad
y residual: \(\mu(T,\rho)=\mu_0(T)+\mu_1(T)\rho+\Delta\mu_r(T,\rho)\).
Se calcula con la densidad de Leachman; no se sustituye por el término diluido solamente.
La corrección de 2022 aclara factores y exponentes y proporciona valores de verificación.

Como comparación de ingeniería se usa Sutherland para viscosidad:
\[
\mu_S(T)=\mu_0(T/T_0)^{3/2}(T_0+S_\mu)/(T+S_\mu),
\quad \mu_0=8.411\times10^{-6}\;\mathrm{Pa\,s},\quad T_0=273\;\mathrm K,
\quad S_\mu=97\;\mathrm K.
\]
Estos parámetros corresponden al H₂ en COMSOL, tabla 5-2. Las aproximaciones se
muestran para comparación; las curvas científicas de referencia son Leachman y Muzny.

El intervalo por defecto de estas dos secciones es **273–1000 K**, sin extrapolación.
La presión común se limita a 200 MPa por el rango publicado de viscosidad (Leachman
abarca hasta 2000 MPa). El rango editable de temperatura se restringe aquí a gas caliente
(273–1000 K). Todas las presiones son **absolutas**, no manométricas ni presión de contacto.


### Settings — Densidad y viscosidad

Instala la dependencia en el kernel del notebook si falta: `%pip install CoolProp`.
También se incluye `config/requirements-properties.txt` para preparar un entorno.
Ejecuta primero Settings de conductividad, que importa NumPy y Plotly.


In [4]:
try:
    import CoolProp
    from CoolProp.CoolProp import PropsSI
except ImportError as error:
    raise ImportError("Install CoolProp in this notebook kernel with: %pip install CoolProp") from error

property_settings = {
    "pressure_Pa": 101325.0,
    "temperature_min_K": 273.0,
    "temperature_max_K": 1000.0,
    "samples": 400,
    "table_temperatures_K": [300, 400, 600, 800, 1000],
    "height": 800,
}
viscosity_sutherland_parameters = {"mu0": 8.411e-6, "T0": 273.0, "Smu": 97.0}
fluid = "HEOS::Hydrogen"  # Normal hydrogen, not ParaHydrogen.
pressure_Pa = float(property_settings["pressure_Pa"])
if not np.isfinite(pressure_Pa) or not 0 < pressure_Pa <= 200e6:
    raise ValueError("Choose a finite absolute pressure in (0, 200e6] Pa.")
if not 273 <= property_settings["temperature_min_K"] < property_settings["temperature_max_K"] <= 1000:
    raise ValueError("Choose 273 <= Tmin < Tmax <= 1000 K; no extrapolation here.")
if property_settings["samples"] < 2:
    raise ValueError("At least two samples are required.")
property_temperature_K = np.linspace(property_settings["temperature_min_K"],
                                    property_settings["temperature_max_K"],
                                    property_settings["samples"])
# Use the EOS gas constant and molar mass for a consistent ideal-gas comparison.
molar_mass = PropsSI("molar_mass", fluid)
gas_constant = PropsSI("gas_constant", fluid)


def reference_property(output, temperature, pressure):
    values = np.asarray(temperature, dtype=float)
    if np.any(~np.isfinite(values)) or np.any((values < 273) | (values > 1000)):
        raise ValueError("Reference comparison requires 273 <= T <= 1000 K.")
    if not np.isfinite(pressure) or not 0 < pressure <= 200e6:
        raise ValueError("Absolute pressure must be in (0, 200e6] Pa.")
    return np.array([PropsSI(output, "T", float(t), "P", pressure, fluid)
                     for t in values.ravel()]).reshape(values.shape)


def density_ideal(temperature, pressure):
    return pressure * molar_mass / (gas_constant * positive_temperature(temperature))


def viscosity_sutherland(temperature):
    T = positive_temperature(temperature)
    mu0, T0, Smu = (viscosity_sutherland_parameters[key] for key in ["mu0", "T0", "Smu"])
    return mu0 * (T/T0)**1.5 * (T0+Smu)/(T+Smu)


rho_reference = reference_property("Dmass", property_temperature_K, pressure_Pa)
rho_ideal = density_ideal(property_temperature_K, pressure_Pa)
mu_reference = reference_property("VISCOSITY", property_temperature_K, pressure_Pa)
mu_sutherland = viscosity_sutherland(property_temperature_K)

# Muzny et al. (2022), verification states: T=40 K; mu in microPa s.
# Tiny positive density approximates the rho=0 dilute limit for the backend.
erratum_density = [1e-9, 50.0, 100.0]
erratum_expected = np.array([1.9772, 5.9905, 49.034])
erratum_computed = np.array([PropsSI("VISCOSITY", "T", 40.0, "Dmass", rho, fluid)
                             for rho in erratum_density]) * 1e6
np.testing.assert_allclose(erratum_computed, erratum_expected, rtol=0, atol=0.0006)
print(f"CoolProp {CoolProp.__version__}; presión absoluta = {pressure_Pa/1e5:.5g} bar")
print("Verificación de viscosidad: los tres estados de la corrección de 2022 concuerdan.")


CoolProp 8.0.0; presión absoluta = 1.0132 bar
Verificación de viscosidad: los tres estados de la corrección de 2022 concuerdan.


### Plot — Densidad
El panel inferior expresa la diferencia del gas ideal respecto a Leachman, no un error experimental.

In [5]:
def property_comparison_plot(reference, approximation, title, ylabel, reference_name, approximation_name):
    figure = make_subplots(rows=2, cols=1, shared_xaxes=True,
                          row_heights=[0.68, 0.32], vertical_spacing=0.13)
    figure.add_trace(go.Scatter(x=property_temperature_K, y=reference, mode="lines",
                               name=reference_name, line={"color": "#1f77b4", "width": 3}), row=1, col=1)
    figure.add_trace(go.Scatter(x=property_temperature_K, y=approximation, mode="lines",
                               name=approximation_name,
                               line={"color": "#d95f02", "width": 3, "dash": "dash"}), row=1, col=1)
    figure.add_trace(go.Scatter(x=property_temperature_K, y=100*(approximation/reference-1),
                               mode="lines", showlegend=False, line={"color": "#6a3d9a"}), row=2, col=1)
    figure.update_layout(
        title={"text": f"{title} — p = {pressure_Pa/1e5:.5g} bar (abs.)", "x": 0.5},
        height=property_settings["height"], template="plotly_white",
        font={"family": "Arial", "size": 15}, hovermode="x unified",
        margin={"t": 140, "b": 70, "l": 95, "r": 50},
        legend={"orientation": "h", "y": 1.10, "x": 0},
    )
    figure.update_yaxes(title_text=ylabel, row=1, col=1)
    figure.update_yaxes(title_text="Diferencia [%]", row=2, col=1)
    figure.update_xaxes(title_text="Temperatura [K]", row=2, col=1)
    figure.add_hline(y=0, line_color="#888888", line_dash="dot", row=2, col=1)
    return figure


density_fig = property_comparison_plot(rho_reference, rho_ideal, "Densidad del hidrógeno",
                                       "ρ [kg/m³]", "Leachman et al. (2009)", "Gas ideal")
density_fig.show()


Plotly figure

### Plot — Viscosidad
Se grafica la viscosidad **dinámica** en μPa·s. Las funciones calculan Pa·s; la conversión gráfica es ×10⁶.

In [6]:
viscosity_fig = property_comparison_plot(mu_reference*1e6, mu_sutherland*1e6,
                                         "Viscosidad dinámica del hidrógeno", "μ [μPa·s]",
                                         "Muzny et al. (2013/2022)", "Sutherland — COMSOL")
viscosity_fig.show()


Plotly figure

### Valores de comparación — Densidad y viscosidad

In [7]:
property_table_T = np.asarray(property_settings["table_temperatures_K"], dtype=float)
property_table_rho = reference_property("Dmass", property_table_T, pressure_Pa)
property_table_mu = reference_property("VISCOSITY", property_table_T, pressure_Pa)
print(f"{'T [K]':>7} {'ρ Leachman':>13} {'ρ ideal':>13} {'Δρ [%]':>10} {'μ Muzny':>13} {'μ Sutherland':>15} {'Δμ [%]':>10}")
for T, rho, mu in zip(property_table_T, property_table_rho, property_table_mu):
    ideal = density_ideal(T, pressure_Pa)
    suth = viscosity_sutherland(T)
    print(f"{T:7.0f} {rho:13.6f} {ideal:13.6f} {100*(ideal/rho-1):10.3f} {mu*1e6:13.5f} {suth*1e6:15.5f} {100*(suth/mu-1):10.3f}")
print("Densidades en kg/m³; viscosidades en μPa·s. Diferencias = 100 × (aproximación/referencia − 1).")


  T [K]    ρ Leachman       ρ ideal     Δρ [%]       μ Muzny    μ Sutherland     Δμ [%]
    300      0.081840      0.081889      0.059       8.93847         9.03019      1.026
    400      0.061387      0.061417      0.048      10.90860        11.10552      1.805
    600      0.040931      0.040944      0.034      14.46714        14.54787      0.558
    800      0.030701      0.030708      0.025      17.70488        17.40395     -1.700
   1000      0.024562      0.024567      0.020      20.72554        19.88835     -4.039
Densidades en kg/m³; viscosidades en μPa·s. Diferencias = 100 × (aproximación/referencia − 1).


## Referencias (APA 7)

Assael, M. J., Assael, J.-A. M., Huber, M. L., Perkins, R. A., & Takata, Y. (2011).
Correlation of the thermal conductivity of normal and parahydrogen from the triple point
to 1000 K and up to 100 MPa. *Journal of Physical and Chemical Reference Data, 40*(3),
Article 033101. https://doi.org/10.1063/1.3606499

COMSOL AB. (n.d.). Sutherland’s law. In *CFD Module user’s guide (version 6.3)*
(Eq. 5-10 and Table 5-3). Retrieved November 6, 2026, from
https://doc.comsol.com/6.3/doc/com.comsol.help.cfd/cfd_ug_fluidflow_high_mach.08.43.html


Leachman, J. W., Jacobsen, R. T., Penoncello, S. G., & Lemmon, E. W. (2009). Fundamental equations of state for parahydrogen, normal hydrogen, and orthohydrogen. *Journal of Physical and Chemical Reference Data, 38*(3), 721–748. https://doi.org/10.1063/1.3160306

Muzny, C. D., Huber, M. L., & Kazakov, A. F. (2013). Correlation for the viscosity of normal hydrogen obtained from symbolic regression. *Journal of Chemical & Engineering Data, 58*(4), 969–979. https://doi.org/10.1021/je301273j

Muzny, C. D., Huber, M. L., & Kazakov, A. F. (2022). Erratum: Correlation for the viscosity of normal hydrogen obtained from symbolic regression. *Journal of Chemical & Engineering Data, 67*, 2855. https://doi.org/10.1021/acs.jced.2c00523

CoolProp. (n.d.). *Hydrogen*. Retrieved November 6, 2026, from https://coolprop.org/fluid_properties/fluids/Hydrogen.html

Los parámetros de Sutherland para viscosidad proceden de la tabla 5-2 de COMSOL citada arriba (conductividad: tabla 5-3). CoolProp es la implementación computacional; las referencias científicas de los modelos son los artículos anteriores.
